In [1]:
from LLMGeometry.datasets import load_dataset_by_name
from LLMGeometry.in_context_learning import ICL_Template, load_ICL_template, list_ICL_templates_in_json
from LLMGeometry import load_model_and_tokenizer
from LLMGeometry.evaluation import run_on_dataframe
from LLMGeometry.utils import generate_random_samples, save_file_with_incremental_suffix
import torch
import pandas as pd
import pickle
from termcolor import colored
from pathlib import Path
import argparse
import json
import sys
from sklearn.metrics import accuracy_score, f1_score
import random
import numpy as np


/gpfs/share/apps/miniconda3/gpu/4.9.2/lib/python3.8/site-packages/requests/__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.8) or chardet (5.2.0)/charset_normalizer (2.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [2]:
MODEL_NAME = 'llama3.1_8b_base'
num_classes = 5
n_relabel = 10

In [4]:
if MODEL_NAME == 'llama3.1_1b_base':
        relabeling_dir = '1B_relabelings'
        relabeling_prefix = '1B_'
elif MODEL_NAME == 'llama3.1_70b_instruct':
    relabeling_dir = '70B_relabelings'
    relabeling_prefix = '70B_'
else:
    relabeling_dir = '7B_relabelings'  # Default fallback
    relabeling_prefix = '7B_'

for n_relabel in range(10, 101, 10):
    relabeling_path = Path(f"{relabeling_dir}/{relabeling_prefix}relabelings_{num_classes}classes_128256toptokens_isensembledFalse_voting_{n_relabel}examples_1runs.pkl")
    print(relabeling_path)
    if not relabeling_path.exists():
        raise ValueError(f"No relabeling file found for n_relabel={n_relabel}. Please run generate_relabelings.py first.")

    print(f"\nLoading relabeling scheme from {relabeling_path}")
    with open(relabeling_path, 'rb') as f:
        relabeling_data = pickle.load(f)
        
    # Verify the relabeling configuration matches
    relabeling_config = relabeling_data['config']
    # if relabeling_config['MODEL_NAME'] != MODEL_NAME:
    #     raise ValueError(f"Relabeling model ({relabeling_config['MODEL_NAME']}) doesn't match current model ({MODEL_NAME})")
    # if relabeling_config['DATASET_NAME'] != DATASET_NAME:
    #     raise ValueError(f"Relabeling dataset ({relabeling_config['DATASET_NAME']}) doesn't match current dataset ({DATASET_NAME})")

    # Get the relabeling (using first run if multiple runs exist)
    new_labels = relabeling_data['relabelings'][0]['labels']
    print("\nUsing relabeling scheme:")
    str_to_print = ""
    for orig_label, (new_token, token_id) in new_labels.items():
        print(f"  {orig_label} -> {new_token} (ID: {token_id})")
        str_to_print += new_token[1:] + ", "
    print(str_to_print)

7B_relabelings/7B_relabelings_5classes_128256toptokens_isensembledFalse_voting_10examples_1runs.pkl

Loading relabeling scheme from 7B_relabelings/7B_relabelings_5classes_128256toptokens_isensembledFalse_voting_10examples_1runs.pkl

Using relabeling scheme:
  E -> Ġtheater (ID: 27803)
  A -> ĠCOLOR (ID: 26493)
  B -> ĠHEALTH (ID: 73725)
  D -> Ġride (ID: 12141)
  C -> ĠOffensive (ID: 76589)
theater, COLOR, HEALTH, ride, Offensive, 
7B_relabelings/7B_relabelings_5classes_128256toptokens_isensembledFalse_voting_20examples_1runs.pkl

Loading relabeling scheme from 7B_relabelings/7B_relabelings_5classes_128256toptokens_isensembledFalse_voting_20examples_1runs.pkl

Using relabeling scheme:
  E -> Ġcinema (ID: 34292)
  A -> ĠBroadcasting (ID: 64560)
  B -> ĠDeng (ID: 92829)
  D -> Ġnut (ID: 10184)
  C -> ĠRugby (ID: 52002)
cinema, Broadcasting, Deng, nut, Rugby, 
7B_relabelings/7B_relabelings_5classes_128256toptokens_isensembledFalse_voting_30examples_1runs.pkl

Loading relabeling scheme fro